# Bubble Bi

### Reading the stock market like a language

## The problem

Every day, a stock leaves a trace: open, high, low, close, volume. Markets repeat certain moods — calm drift, panic, sharp reversal — but nobody labels them, and there is no dictionary of them.

**The idea:** let the machine build that dictionary itself.

We compress each stock-day into a single **token** — one word out of 512 that the machine invents on its own. A stock's history then becomes a sentence:

```
AAPL: ... 147  147  391  208  208  63  →  what comes next?
```

From there it is the same trick as ChatGPT: read the sentence, predict the next word.

**Three stages:**

| | | |
|---|---|---|
| 1. **Tokenizer** | turns each stock-day into one token | ✅ built |
| 2. **Predictor** | a GPT that predicts the next token | ✅ built |
| 3. **Trader** | acts on it: long / short / flat | ⬜ not built |

> ⚠️ **The one rule:** nothing may ever look into the future. Break it and the model looks brilliant while losing real money. A test enforces it.

Run the cells below in order.

---
## 1. Settings

Every knob in the project is here. Change these, re-run the notebook, get a different result. Nothing is hidden in a config file.

In [ ]:
SETTINGS = dict(

    # ── Which companies, and how far back ───────────────────────────────
    tickers = ["AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "JPM", "V", "JNJ", "WMT",
               "PG", "HD", "BAC", "XOM", "CVX", "KO", "PEP", "DIS", "CSCO", "INTC",
               "VZ", "T", "MRK", "PFE", "ABT", "NKE", "MCD", "CAT", "BA", "IBM"],
    start   = "2010-01-01",

    # ── The tokenizer: how a stock-day becomes one "word" ───────────────
    vocabulary     = 512,   # how many different market words may exist
    days_per_token = 4,     # how many days of history each word summarises
    market_days    = 5,     # how many days of the WHOLE market it also sees
    model_size     = 128,   # brain size of the tokenizer

    # ── The predictor: the GPT that reads the sentences ─────────────────
    sentence_length  = 64,  # how many past tokens it reads before guessing
    predictor_layers = 4,   # how deep the GPT is

    # ── Training ────────────────────────────────────────────────────────
    steps         = 2000,   # how long to train (raise this on a GPU)
    batch_size    = 256,
    learning_rate = 1e-4,
    seed          = 42,     # same seed = same result, every time
)

---
## 2. Get the code

Downloads the project and installs what it needs. Safe to re-run — it skips anything already there.

In [ ]:
import os, subprocess, sys

REPO   = "https://github.com/hockper/Quant-AI-2026.git"
BRANCH = "notebook-first"

if not os.path.exists("bubble.py"):                 # not here yet -> fetch it
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "bubble-bi"], check=True)
    os.chdir("bubble-bi")

sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                "backup/requirements.txt"], check=True)

import bubble

config = bubble.make_config(**SETTINGS)
print(bubble.describe(config))